# 05 — Advanced Models: Gradient Boosting (LightGBM and XGBoost)

**Goal (game plan Week 7):** try gradient-boosted trees with light tuning and compare them with the earlier models.
**Author:** Ethan Chung — IDX Exchange Data Science, Fall 2026

Gradient boosting builds trees one after another, each one correcting the errors left by the trees before it. A Random Forest instead averages many independent trees. Both models here use the same `week6` features, leakage-safe pipeline and procedure as every other model (`src/models.py`, IDX Best Practices §01, §09):
1. **Light tuning on the 2026-03 validation month only:** training window (3 / 6 / 12 / 24 months) × 4 settings of tree size and minimum leaf size. Learning rate (0.03), tree count (2,000) and 80% row / column subsampling are fixed.
2. **Test once** on 2026-04 after refitting on the months right before it.
3. **Rolling backtest** on 2026-03, 2026-02, 2026-01.

**Run `python src/models.py` first** (about 35 minutes for all five models; `python src/models.py lightgbm xgboost` runs only the two new ones in about 25 minutes). This notebook only loads the saved results. LightGBM and XGBoost on a Mac need the OpenMP runtime: `brew install libomp`.

## 1. Setup

In [ ]:
import os
import sys

import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.insert(0, os.path.abspath("../src"))
import evaluation
import models
import preprocessing

pd.set_option("display.float_format", lambda x: f"{x:,.2f}")
FEATURE_SET = "week6"
NAMES = list(models.MODELS)
BOOSTED = ["lightgbm", "xgboost"]
LABELS = {"baseline_linear_regression": "Linear Regression", "decision_tree": "Decision Tree",
          "random_forest": "Random Forest", "lightgbm": "LightGBM", "xgboost": "XGBoost"}
METRICS = ["R2", "MdAPE", "MAPE", "MAE", "RMSE", "within_10pct", "median_pct_error"]

clean = preprocessing.load_clean("../data/processed/sfr_clean.parquet")
metrics = pd.concat([pd.read_csv(f"../docs/metrics_{models.artifact_stem(n, FEATURE_SET)}.csv") for n in NAMES],
                    ignore_index=True)

# Reload each saved model and score its own test set (test month, cutoffs from its training window)
results = {}
for n in NAMES:
    path = f"../models/{models.artifact_stem(n, FEATURE_SET)}.joblib"
    art = joblib.load(path)
    _, test, _ = preprocessing.make_final_split(clean, art["train_info"]["train_months"])
    results[n] = {"model": art["model"], "params": art["params"], "info": art["train_info"], "test": test,
                  "pred": models.predict_dollars(art["model"], test), "file_mb": os.path.getsize(path) / 1e6}
{LABELS[n]: (r["info"]["train_months"], r["params"]) for n, r in results.items()}

## 2. Light tuning on the validation month

Every (window, setting) combination is fit on the months before 2026-03 and scored on 2026-03. The test month is not used.

In [ ]:
val = metrics[metrics["stage"] == "validation (selection)"].copy()
val["params"] = val["params"].astype(str)
for n in BOOSTED:
    grid = val[val["model"] == n].pivot(index="params", columns="train_months", values="MdAPE")
    print(f"{LABELS[n]}: validation MdAPE (%) by setting and window")
    display(grid)

In [ ]:
# Best validation MdAPE per window, every model
by_window = val.groupby(["model", "train_months"])["MdAPE"].min().unstack("model")[NAMES].rename(columns=LABELS)
ax = by_window.plot(marker="o", figsize=(7, 3.5), title="Validation MdAPE by training window (2026-03)")
ax.set_xlabel("training window (months)"); ax.set_ylabel("MdAPE (%)"); ax.set_xticks(models.WINDOWS)
plt.tight_layout(); plt.show()
best_val = val.loc[val.groupby("model")["MdAPE"].idxmin(), ["model", "train_months", "params", "R2", "MdAPE", "MAPE"]]
best_val.set_index("model").loc[NAMES].rename(index=LABELS)

## 3. Side-by-side test results (2026-04)

Linear Regression baseline next to the tree ensembles, same table (IDX Best Practices §10).

In [ ]:
test_table = metrics[metrics["stage"] == "test"].set_index("model").loc[NAMES, ["train_months"] + METRICS]
test_table["model file (MB)"] = [results[n]["file_mb"] for n in NAMES]
test_table.rename(index=LABELS)

### Is the improvement more than noise?

Paired bootstrap on the test homes every model scored: each round resamples homes and recomputes both MdAPEs on the same sample. If the 95% interval for the improvement excludes 0, the gain is real.

In [ ]:
common = sorted(set.intersection(*(set(r["test"]["ListingKey"]) for r in results.values())))
aligned = {n: pd.Series(r["pred"], index=r["test"]["ListingKey"].to_numpy()).loc[common] for n, r in results.items()}
y = clean.set_index("ListingKey").loc[common, "ClosePrice"]
pairs = [("baseline_linear_regression", "lightgbm"), ("baseline_linear_regression", "xgboost"),
         ("random_forest", "lightgbm"), ("random_forest", "xgboost"), ("lightgbm", "xgboost")]
rows = [{"compared with": LABELS[a], "model": LABELS[b], **evaluation.paired_bootstrap_mdape(y, aligned[a], aligned[b])}
        for a, b in pairs]
print(f"{len(common):,} test homes scored by all models")
pd.DataFrame(rows).rename(columns={"mdape_a": "reference MdAPE", "mdape_b": "model MdAPE"})

## 4. Stability: rolling backtest

In [ ]:
hist = metrics[metrics["stage"].isin(["test", "backtest"])][["model", "test_month", "R2", "MdAPE"]]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for n in NAMES:
    h = hist[hist["model"] == n].sort_values("test_month")
    axes[0].plot(h["test_month"], h["R2"], marker="o", label=LABELS[n])
    axes[1].plot(h["test_month"], h["MdAPE"], marker="o", label=LABELS[n])
axes[0].set_title("R² by test month"); axes[1].set_title("MdAPE (%) by test month")
axes[1].set_ylim(0, None); axes[0].legend()
plt.tight_layout(); plt.show()
hist.pivot(index="test_month", columns="model", values="MdAPE")[NAMES].rename(columns=LABELS)

## 5. Where the boosted models gain

Scored on the test homes every model kept (the 12- and 24-month windows cut slightly different price outliers), so the price bands are identical for every model.

In [ ]:
SHOW = ["baseline_linear_regression", "random_forest", "lightgbm", "xgboost"]
band = evaluation.price_bands(y)
county = clean.set_index("ListingKey").loc[common, "CountyOrParish"]
band_tables = [evaluation.metrics_by_group(y, aligned[n], band).assign(model=LABELS[n]) for n in SHOW]
bands = pd.concat(band_tables)
bands["band"] = bands["group"].str.split(" ").str[0]
order = [LABELS[n] for n in SHOW]
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
bands.pivot(index="band", columns="model", values="MdAPE")[order].plot(kind="bar", ax=axes[0], rot=0, title="MdAPE (%) by price quintile")
bands.pivot(index="band", columns="model", values="median_pct_error")[order].plot(kind="bar", ax=axes[1], rot=0, title="Median signed error (%) by quintile (> 0 = over-valued)")
axes[1].axhline(0, color="black", linewidth=0.8)
plt.tight_layout(); plt.show()
bands.pivot(index="group", columns="model", values=["MdAPE", "median_pct_error"])

In [ ]:
county_tables = [evaluation.metrics_by_group(y, aligned[n], county, min_n=200).assign(model=LABELS[n]) for n in SHOW]
pd.concat(county_tables).pivot(index="group", columns="model", values="MdAPE")[order].sort_values("LightGBM")

## 6. What the boosted model relies on

Gain importance: the total improvement in the training loss from splits on each feature, as a share of the total. Encoded columns are grouped back to the input they came from (e.g. all county dummies → `CountyOrParish`).

In [ ]:
def grouped_gain(name):
    pipe = results[name]["model"]
    names = pipe.named_steps["preprocess"].get_feature_names_out()
    est = pipe.named_steps["model"]
    gain = (est.booster_.feature_importance(importance_type="gain") if name == "lightgbm"
            else np.array([est.get_booster().get_score(importance_type="total_gain").get(f"f{i}", 0.0) for i in range(len(names))]))
    # "County_Orange", "missingindicator_LotSizeSquareFeet" -> the input column they encode
    inputs = sorted(pipe.named_steps["preprocess"].feature_names_in_, key=len, reverse=True)
    source = pd.Series(names).str.removeprefix("missingindicator_")
    source = source.map(lambda s: next((c for c in inputs if s == c or s.startswith(c + "_")), s))
    return (pd.Series(gain, index=source).groupby(level=0).sum() / gain.sum() * 100)

imp = pd.DataFrame({LABELS[n]: grouped_gain(n) for n in BOOSTED}).sort_values("LightGBM", ascending=False)
imp.head(15).round(1)

## 7. Practical comparison for the Week 9 app

In [ ]:
pd.DataFrame({
    "test MdAPE": [test_table.loc[n, "MdAPE"] for n in NAMES],
    "model file (MB)": [results[n]["file_mb"] for n in NAMES],
}, index=[LABELS[n] for n in NAMES])

## 8. Findings

| Test 2026-04 | Window | R² | MdAPE | MAPE | MAE | Within 10% | Backtest MdAPE (2026-01 – 03) | Model file |
|---|---|---|---|---|---|---|---|---|
| Linear Regression | 24 mo | 0.827 | 11.0% | 15.1% | $205k | 46% | 10.8–11.2% | 0.1 MB |
| Decision Tree | 24 mo | 0.845 | 9.3% | 13.8% | $189k | 53% | 9.0–9.5% | 0.7 MB |
| Random Forest | 24 mo | 0.897 | 7.5% | 11.1% | $153k | 61% | 7.4–7.7% | 1,106 MB |
| **LightGBM** | 12 mo | **0.910** | **7.15%** | 10.7% | **$145k** | 63% | 6.9–7.3% | 19 MB |
| **XGBoost** | 12 mo | 0.908 | 7.16% | 10.7% | $146k | 63% | 6.9–7.2% | 35 MB |

- **Gradient boosting is the best model family.** Both boosted models beat the Random Forest by about 0.37 MdAPE points (95% paired-bootstrap interval 0.20–0.52), so the gain is more than noise. Versus the Linear Regression baseline the gain is 3.85 points (3.58–4.10).
- **LightGBM and XGBoost are tied:** 6.91% vs 6.88% on validation and 7.15% vs 7.16% on test (difference −0.01, interval −0.12 to +0.10). **LightGBM is the pick** for the final model because its saved file is about half the size (19 MB vs 35 MB, and 1/57 of the Random Forest).
- **Stable over time:** both are lower than the Random Forest in every one of the four test months.
- **Both chose a 12-month window**, unlike the other models (24). Boosting fits the training data closely, so the most recent year describes the current market better; 24 months was a close second on validation.
- **Both chose the largest trees in the grid** (255 leaves / depth 10). A slightly larger grid might help a little more; that is the next tuning step, still on validation only.
- **Luxury homes improve the most:** in the top price quintile LightGBM under-values the typical home by 4.0%, versus 7.0% for the Random Forest and 9.6% for the baseline (MdAPE 10.4% vs 11.2% vs 15.7%). LightGBM is at or below the Random Forest in every quintile and every county with 200+ test homes; Alameda is still the hardest county (9.5%).
- **What it relies on:** location dominates (ZIP code 55–60% of gain, city 6–15%), then living area (14%). The school-district features add 3–6%. Target-encoded location columns carry a lot of the price signal, so gain importance favors them.

**Strengths and weaknesses**

| Model | Strengths | Weaknesses |
|---|---|---|
| Random Forest | Few settings to tune; averages many trees, hard to overfit | 1.1 GB file; slightly worse on every metric than boosting |
| LightGBM / XGBoost | Best accuracy, smallest error on luxury homes, small files, fits in under a minute | More settings to tune; trees built in sequence can overfit if the learning rate or tree count is pushed |

**Next (Week 8):** full evaluation notebook (`06_evaluation.ipynb`) with MAPE / MdAPE by price band and region for the final LightGBM model, plus `docs/metrics_summary.csv`.